<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [ ]:
!pip3 install folium
!pip3 install wget
!pip3 install pandas

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=20aa23ad7eb849054c2c23bc36486e68a72f418a9f9934223fab967e546bc5d3
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [ ]:
import folium
import wget
import pandas as pd

In [ ]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [ ]:
# Download and read the `spacex_launch_geo.csv`
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df=pd.read_csv(spacex_csv_file)

Now, you can take a look at what are the coordinates for each site.


In [ ]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [ ]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [ ]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


_TODO:_  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [ ]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label
for index, record in launch_sites_df.iterrows():
  coordinate = [record['Lat'], record['Long']]

  circle = folium.Circle(coordinate, radius=80, color='#d35400', fill=True).add_child(folium.Popup(f'{record['Launch Site']}'))
  # Create a blue circle at launch coordinate with a icon showing its name
  marker = folium.map.Marker(
      coordinate,
      # Create an icon as a text label
      icon=DivIcon(
          icon_size=(20,20),
          icon_anchor=(0,0),
          html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % f'{record['Launch Site']}',
          )
      )
  site_map.add_child(circle)
  site_map.add_child(marker)
site_map



In [ ]:
equator_line = folium.PolyLine(
    locations=[[0, -180], [0, 180]],
    color='red',
    weight=2,
    opacity=0.8,
    tooltip='Equator'
)
site_map.add_child(equator_line)
display(site_map)

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:
- Are all launch sites in proximity to the Equator line?
- Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


### Are all launch sites in proximity to the Equator line?

In [ ]:
import pandas as pd
# Define a function to calculate distance to equator (roughly 111km per degree)
def dist_to_equator(lat):
    return abs(lat) * 111.32

proximity_df = launch_sites_df[['Launch Site', 'Lat']].copy()
proximity_df['Distance to Equator (KM)'] = proximity_df['Lat'].apply(dist_to_equator)

display(proximity_df.sort_values(by='Lat'))

,Launch Site,Lat,Distance to Equator (KM)
0,CCAFS LC-40,28.562302,3179.555455
1,CCAFS SLC-40,28.563197,3179.655110
2,KSC LC-39A,28.573255,3180.774699
3,VAFB SLC-4E,34.632834,3855.327099


### Findings:
- **Proximity:** While these are some of the southernmost points in the continental US, they are technically in the mid-latitudes (28°N to 34°N), not immediate proximity to the Equator.
- **Reasoning:** Launch sites are placed as far south as possible to leverage the Earth's higher tangential velocity at the equator, which reduces the amount of fuel needed to reach orbit.

### Are all launch sites in very close proximity to the coast?

### Findings - Proximity to Coastline:
- **Proximity:** All launch sites are in very close proximity to the coastline (usually within 1-5 KM).
- **Reasoning:** Safety is the primary factor. Launching over the ocean minimizes the risk to human life and property in case of a launch failure or during the separation of rocket stages.

# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [ ]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [ ]:
marker_cluster = MarkerCluster()


_TODO:_ Create a new column in `launch_sites` dataframe called `marker_color` to store the marker colors based on the `class` value


In [ ]:
# Function to assign color to launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


_TODO:_ For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [ ]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed,
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
for index, record in spacex_df.iterrows():
    # TODO: Create and add a Marker cluster to the site map
    coordinate = [record['Lat'], record['Long']]
    marker = folium.Marker(coordinate,
      # Create an icon as a text label
      icon=folium.Icon(color='white', icon_color=record['marker_color'])
    )
    marker_cluster.add_child(marker)

site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [ ]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topleft',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [ ]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

_TODO:_ Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [ ]:
spacex_df.head()

,Launch Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


In [ ]:
spacex_df['Launch Site'].unique()

array(['CCAFS LC-40', 'VAFB SLC-4E', 'KSC LC-39A', 'CCAFS SLC-40'],
      dtype=object)

In [ ]:
vafb_lat, vafb_long = spacex_df[spacex_df['Launch Site'] == 'VAFB SLC-4E'][['Lat', 'Long']].values[0]
ccafslc_lat, ccafslc_long = spacex_df[spacex_df['Launch Site'] == 'CCAFS LC-40'][['Lat', 'Long']].values[0]
ksc_lat, ksc_long = spacex_df[spacex_df['Launch Site'] == 'KSC LC-39A'][['Lat', 'Long']].values[0]
ccafsslc_lat, ccafsslc_long = spacex_df[spacex_df['Launch Site'] == 'CCAFS SLC-40'][['Lat', 'Long']].values[0]

In [ ]:
vafb_lat, vafb_long

(np.float64(34.63283416), np.float64(-120.6107455))

In [ ]:
ccafslc_lat, ccafslc_long

(np.float64(28.56230197), np.float64(-80.57735648))

In [ ]:
ksc_lat, ksc_long

(np.float64(28.57325457), np.float64(-80.64689529))

In [ ]:
ccafsslc_lat, ccafsslc_long

(np.float64(28.56319718), np.float64(-80.57682003))

In [ ]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
vafb_lat, vafb_long = spacex_df[spacex_df['Launch Site'] == 'VAFB SLC-4E'][['Lat', 'Long']].values[0]
ccafslc_lat, ccafslc_long = spacex_df[spacex_df['Launch Site'] == 'CCAFS LC-40'][['Lat', 'Long']].values[0]
ksc_lat, ksc_long = spacex_df[spacex_df['Launch Site'] == 'KSC LC-39A'][['Lat', 'Long']].values[0]
ccafsslc_lat, ccafsslc_long = spacex_df[spacex_df['Launch Site'] == 'CCAFS SLC-40'][['Lat', 'Long']].values[0]

nearest_proximity_map = {
    "VAFB SLC-4E": {
        "coastline": [34.6328, -120.6270],
        "railway": [34.6328, -120.6250],
        "highway": [34.6844, -120.6035],
        "city": [34.6391, -120.4579]
    },
    "CCAFS LC-40": {
        "coastline": [28.5623, -80.5650],
        "railway": [28.5623, -80.8075],
        "highway": [28.5623, -80.5825],
        "city": [28.3922, -80.6077]
    },
    "KSC LC-39A": {
        "coastline": [28.5732, -80.5695],
        "railway": [28.5732, -80.6540],
        "highway": [28.5732, -80.6465],
        "city": [28.5732, -80.7987]
    },
    "CCAFS SLC-40": {
        "coastline": [28.5632, -80.5679],
        "railway": [28.5721, -80.5853],
        "highway": [28.5632, -80.5707],
        "city": [28.4017, -80.6042]
    }
}

# distance_coastline = calculate_distance(vafb_lat, vafb_long, coastline_lat, coastline_lon)

_TODO:_ After obtained its coordinate, create a `folium.Marker` to show the distance


In [ ]:
launch_site_distances = {}

# We'll use the unique launch sites from the dataframe as the reference
for index, row in launch_sites_df.iterrows():
    site_name = row['Launch Site']
    site_lat = row['Lat']
    site_long = row['Long']

    # Initialize the sub-dictionary for this site
    launch_site_distances[site_name] = {}

    # Get the proximity coordinates from our map
    proximities = nearest_proximity_map.get(site_name, {})

    for feature, coords in proximities.items():
        dist = calculate_distance(site_lat, site_long, coords[0], coords[1])
        launch_site_distances[site_name][feature] = dist

# Display the calculated distances
import json
print(json.dumps(launch_site_distances, indent=4))

{
    "CCAFS LC-40": {
        "coastline": 1.207140537414406,
        "railway": 22.483387281893844,
        "highway": 0.5024854964630959,
        "city": 19.151594824951985
    },
    "CCAFS SLC-40": {
        "coastline": 0.8714163338406976,
        "railway": 1.291063393277557,
        "highway": 0.5978785379294546,
        "city": 18.161658860644412
    },
    "KSC LC-39A": {
        "coastline": 7.560188546531498,
        "railway": 0.6940342891174887,
        "highway": 0.03908718549183711,
        "city": 14.828704683165578
    },
    "VAFB SLC-4E": {
        "coastline": 1.4876350486928143,
        "railway": 1.3045934712183154,
        "highway": 5.773841251191335,
        "city": 14.005412010226719
    }
}


In [ ]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property
# for example
distance_marker = folium.Marker(
   coordinate,
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
       )
   )

_TODO:_ Draw a `PolyLine` between a launch site to the selected coastline point


In [ ]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
# lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

In [ ]:
for index, row in launch_sites_df.iterrows():
    site_name = row['Launch Site']
    site_lat = row['Lat']
    site_long = row['Long']

    # Get coastline coordinates and distance
    coast_coords = nearest_proximity_map[site_name]['coastline']
    distance = launch_site_distances[site_name]['coastline']

    # Create a distance marker
    distance_marker = folium.Marker(
        coast_coords,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
        )
    )
    site_map.add_child(distance_marker)

    # Create and add the PolyLine
    line = folium.PolyLine(locations=[[site_lat, site_long], coast_coords], weight=1)
    site_map.add_child(line)

display(site_map)

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


_TODO:_ Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [ ]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site
feature_colors = {
    'coastline': 'blue',
    'railway': 'green',
    'highway': 'red',
    'city': 'black'
}

for index, row in launch_sites_df.iterrows():
    site_name = row['Launch Site']
    site_lat = row['Lat']
    site_long = row['Long']

    # Iterate through all features for this site
    proximities = nearest_proximity_map.get(site_name, {})

    for feature, feature_coords in proximities.items():
        distance = launch_site_distances[site_name][feature]
        color = feature_colors.get(feature, 'gray')

        # Create a distance marker at the feature coordinate
        distance_marker = folium.Marker(
            feature_coords,
            icon=DivIcon(
                icon_size=(20,20),
                icon_anchor=(0,0),
                html='<div style="font-size: 12; color:{};"><b>{:10.2f} KM</b></div>'.format(color, distance),
            )
        )
        site_map.add_child(distance_marker)

        # Draw a PolyLine between site and feature with specific color
        line = folium.PolyLine(locations=[[site_lat, site_long], feature_coords], weight=2, color=color)
        site_map.add_child(line)

display(site_map)

After you plot distance lines to the proximities, you can answer the following questions easily:
- Are launch sites in close proximity to railways?
- Are launch sites in close proximity to highways?
- Are launch sites in close proximity to coastline?
- Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


### **The nearest railways, highways, coastlines, and cities to Vandenberg Space Force Base Space Launch Complex 4E (SLC-4E), located at 34.6328, -120.6107:**

1. Nearest Coastline: The Pacific Ocean
SLC-4E sits right on the edge of the California coast, providing a safe, open-water trajectory for southern and polar orbital launches. The nearest point of the coastline is located directly west of the launch pad.

Coordinates: 34.6328, -120.6270

Approximate Distance: 0.9 miles (1.5 km) West.

2. Nearest Railway: Union Pacific Railroad (Coast Line)
The Union Pacific Railroad, which also carries the Amtrak Coast Starlight and Pacific Surfliner, hugs the coastline and cuts directly through Vandenberg Space Force Base. It runs between the launch pad and the beach.

Coordinates: 34.6328, -120.6250

Approximate Distance: 0.8 miles (1.3 km) West.

3. Nearest Highway: California State Route 246 (W Ocean Ave)
While there are plenty of restricted military access roads on the base itself, the nearest major public highway is State Route 246. This highway runs east-to-west and officially terminates at Surf Beach, located just north of the launch complex.

Coordinates: 34.6844, -120.6035 (Near the Surf Beach terminus)

Approximate Distance: 3.5 miles (5.6 km) North.

4. Nearest City: Lompoc, California
Lompoc is the closest incorporated city to Vandenberg Space Force Base and serves as the primary residential hub for many of the base's personnel.

Coordinates: 34.6391, -120.4579 (City Center)

Approximate Distance: 8.7 miles (14 km) East.

### **The nearest coastlines, highways, railways, and cities to Cape Canaveral Space Force Station Space Launch Complex 40 (SLC-40), located at 28.56230197, -80.57735648:**

1. Nearest Coastline: The Atlantic Ocean
SLC-40 is situated on the eastern edge of the Cape Canaveral peninsula. The Atlantic Ocean coastline is located directly east of the launch pad, providing a clear flight path for launches aiming for equatorial and mid-inclination orbits.

Coordinates: 28.5623, -80.5650

Approximate Distance: 0.8 miles (1.3 km) East.

2. Nearest Highway: Samuel C. Phillips Parkway / US Route 1
Samuel C. Phillips Parkway is the primary multi-lane road running north-to-south directly through the Space Force Station, passing just west of SLC-40.

Coordinates (Phillips Pkwy): 28.5623, -80.5825

Approximate Distance: 0.3 miles (0.5 km) West.

For the nearest major public highway outside the restricted base, US Route 1 runs parallel to the coast on the mainland across the Indian River.

Coordinates (US Route 1): 28.5623, -80.8030

Approximate Distance: 14 miles (22.5 km) West.

3. Nearest Railway: Florida East Coast (FEC) Railway
While NASA previously operated its own industrial short-line railroad on the Cape (which was officially retired in 2015), the closest active commercial railway is the Florida East Coast Railway mainline. This track runs north-to-south through the mainland city of Titusville.

Coordinates: 28.5623, -80.8075

Approximate Distance: 14.2 miles (22.8 km) West.

4. Nearest City: Cape Canaveral, Florida
The City of Cape Canaveral is the closest municipality, located just south of the Space Force Station and Port Canaveral. It serves as a major residential and commercial hub for space industry personnel.

Coordinates: 28.3922, -80.6077 (City Center)

Approximate Distance: 11.8 miles (19 km) South.

### **The nearest coastlines, highways, railways, and cities to Cape Kennedy Space Center Launch Complex 39A (KSC LC-39A), located at 28.56230197, -80.64689529:**
1. Nearest Highway: Kennedy Parkway North (State Road 3)
Because this dataset point is located further inland, it is practically sitting right on top of Kennedy Parkway North, the main north-south arterial road connecting the KSC facilities.

Coordinates: 28.5732, -80.6465 (Immediately adjacent)

Approximate Distance: Less than 0.1 miles (0.1 km) East.

2. Nearest Railway: NASA Railroad
NASA’s internal short-line railroad (used primarily for transporting heavy solid rocket booster segments) runs just west of this inland coordinate, near the Vehicle Assembly Building.

Coordinates: 28.5732, -80.6540

Approximate Distance: 0.4 miles (0.7 km) West.

3. Nearest Coastline: The Atlantic Ocean
Since this point is significantly further inland than the actual LC-39A pad, the distance to the eastern Atlantic coastline (over the Banana River and the Cape Canaveral peninsula) is much greater.

Coordinates: 28.5732, -80.5695

Approximate Distance: 4.7 miles (7.6 km) East.

4. Nearest City: Titusville, Florida
Titusville remains the closest city. Measuring from this specific inland coordinate across the Indian River Lagoon to the nearest mainland shoreline of Titusville:

Coordinates: 28.5732, -80.7987 (Nearest city limit / shoreline)

Approximate Distance: 9.2 miles (14.8 km) West.

### **The nearest coastlines, highways, railways, and cities to the specific coordinates for Cape Canaveral Space Force Station Space Launch Complex 40 (CCAFS SLC-40) at 28.56319718, -80.57682003:**

1. Nearest Coastline: The Atlantic Ocean
The launch pad is located very close to the eastern shore of the Cape Canaveral peninsula. The closest point on the coastline is located almost directly east of the launch pad.

Approximate Coordinates: 28.5632, -80.5679

Approximate Distance: 0.54 miles (0.87 km) East.

2. Nearest Highway: Samuel C. Phillips Parkway
Samuel C. Phillips Parkway is the main north-south arterial road running through the Cape Canaveral Space Force Station. The closest point on this road sits just east of the SLC-40 perimeter.

Approximate Coordinates: 28.5632, -80.5707

Approximate Distance: 0.37 miles (0.60 km) East.

3. Nearest Railway: NASA Railroad / Florida East Coast (FEC) Railway
If you are looking for the absolute closest railroad tracks (including inactive government lines often used in spatial datasets), the old NASA Railroad system has tracks located to the northwest of the pad.

Approximate Coordinates (NASA Railroad): 28.5721, -80.5853

Approximate Distance: 0.80 miles (1.28 km) Northwest.

For the nearest active commercial railway, the Florida East Coast (FEC) Railway runs north-to-south on the mainland in Titusville.

Approximate Coordinates (FEC Railway): 28.5632, -80.8076

Approximate Distance: 14.1 miles (22.6 km) West.

4. Nearest City: Cape Canaveral, Florida
The City of Cape Canaveral is the closest incorporated municipality, located to the south of the Space Force Station and Port Canaveral.

Approximate Coordinates: 28.4017, -80.6042 (City Center)

Approximate Distance: 11.2 miles (18.1 km) South.

## Proximity to Railways Analysis



### **Are launch sites in close proximity to railways?**

**Findings on Proximity to Railways:**

Based on the analysis of the launch site coordinates and the `launch_site_distances` data, we can observe the following regarding proximity to railway infrastructure:

*   **VAFB SLC-4E:** The Union Pacific Railroad (Coast Line) is approximately **1.3 km** away.
*   **KSC LC-39A:** The NASA Railroad is approximately **0.7 km** away.
*   **CCAFS SLC-40:** The NASA Railroad/industrial tracks are approximately **1.28 km** away.
*   **CCAFS LC-40:** While the mainland commercial railway is further away, historical industrial spurs for the Cape Canaveral Space Force Station were designed to support heavy transport.

**Conclusion:** Most launch sites are located within **1-2 km** of a railway.

**Reasoning:**
Railways are essential for space launch logistics. Rocket components, such as heavy first-stage boosters, large rocket engines, and massive propellant tanks, are often too large or heavy to be transported efficiently by standard road vehicles or air freight. Proximity to a railway allows these critical components to be transported directly from manufacturing facilities (like those in California or Texas) to the processing hangars at the launch site with minimal handling risk.

## Proximity to Highways Analysis


### Are all launch sites in close proximity to highways?

**Findings:**
- **Proximity:** Most launch sites are located within very close proximity to major roads or highways, typically ranging from **0.04 km to 5.8 km**.
    - **KSC LC-39A** is exceptionally close (~0.04 km) to Kennedy Parkway North (State Road 3).
    - **CCAFS LC-40** is approximately 0.5 km from Samuel C. Phillips Parkway.
    - **VAFB SLC-4E** is roughly 5.8 km from California State Route 246.

**Reasoning:**
Highways and major roads are essential for the daily operations of a launch site. They provide the necessary infrastructure for:
1. **Logistical Support:** Easy access for trucks delivering smaller equipment, components, and general supplies.
2. **Personnel Transport:** Efficient commuting for the hundreds of engineers, technicians, and security staff required for launch operations.
3. **Fuel and Commodity Delivery:** While large rocket stages may arrive via rail or sea, many consumables like propellants, oxidizers, and high-pressure gases are often transported via specialized road tankers.

## Proximity to Coastlines Analysis


### Are launch sites in close proximity to coastlines?

Based on the proximity analysis conducted above, all SpaceX launch sites are located in very close proximity to the coast, with distances to the nearest coastline ranging from approximately **0.8 km to 7.6 km**.

#### Findings - Proximity to Coastline:
- **Safety:** The primary reasoning for this proximity is launch safety. Launching over open water minimizes the risk to human life and property in the event of a launch failure. Additionally, the ocean provides a safe area for the planned separation and disposal of rocket stages.
- **Launch Corridors:** Coastal locations provide unobstructed 'launch corridors' for different orbital inclinations. Sites on the Atlantic coast (Cape Canaveral and Kennedy Space Center) are ideal for equatorial and mid-inclination orbits, while the Pacific coast (Vandenberg Space Force Base) is suited for polar orbital trajectories.

## Distance from Cities Analysis


### Do launch sites keep a certain distance from cities?

Based on the proximity analysis, launch sites are typically located between **14 km and 23 km** away from major residential centers such as Lompoc, Titusville, and the city of Cape Canaveral.

**Findings:**
- **Safety Buffer:** The distance serves as a critical safety buffer to protect large populations from potential launch failures, including debris or explosions during the ascent phase.
- **Acoustic Impact:** Rockets generate extreme noise pollution and sonic booms. Placing sites several kilometers away from cities helps mitigate the structural and physiological impact of these sound waves.
- **Controlled Access:** Maintaining a distance from urban centers allows for large restricted zones (exclusion zones) during launch windows, ensuring that if a stage separation or failure occurs, the impact is contained within unpopulated land or open water.

## Summary of Findings



### Summary of Geographical Patterns

Based on the visual analysis and distance calculations across all SpaceX launch sites (CCAFS LC-40, CCAFS SLC-40, KSC LC-39A, and VAFB SLC-4E), several consistent geographical patterns emerge:

*   **Extreme Proximity to Coastlines (0.8 - 7.6 km):** All sites are situated immediately adjacent to the ocean (Atlantic or Pacific). This optimizes for safety by allowing for water-based trajectories, ensuring that rocket stages and potential debris fall into the sea rather than populated areas.
*   **Logistical Integration (Railways & Highways):**
    *   **Railways (0.7 - 1.3 km):** Most sites are within very close proximity to railway lines, which are essential for transporting heavy and oversized rocket components like boosters and fuel tanks.
    *   **Highways (0.04 - 5.8 km):** Close proximity to major roads and highways facilitates the rapid movement of personnel, smaller equipment, and general supplies.
*   **Safety Buffers from Cities (14 - 23 km):** While sites are close to infrastructure, they maintain a significant distance from major residential hubs (e.g., Titusville, Lompoc). This provides a critical safety buffer to minimize the impact of noise, vibrations, and potential launch failures on civilian populations.

**Conclusion:** These site selections demonstrate a strategic optimization that balances **safety** (coasts and city buffers), **logistical efficiency** (rail and road access), and **orbital flexibility** (leveraging southern latitudes for equatorial or polar launches).

## Final Task


## Summary:

### Q&A

**Are SpaceX launch sites in close proximity to railways?**
Yes, most launch sites are located within **1.3 km** of a railway (e.g., KSC LC-39A is only ~0.7 km away). This proximity is vital for transporting oversized rocket components like boosters and propellant tanks that cannot be moved by road.

**Are all launch sites in close proximity to highways?**
Yes, launch sites are strategically placed near major roads, with distances ranging from **0.04 km to 5.8 km**. This ensures easy access for personnel, delivery of smaller equipment, and transport of consumables via road tankers.

**Are launch sites in close proximity to coastlines?**
Yes, all sites are between **0.8 km and 7.6 km** from the ocean. This allows for safe launch trajectories over water, minimizing risks to populated areas and providing "launch corridors" for different orbital paths (equatorial from the East Coast and polar from the West Coast).

**Do launch sites keep a certain distance from cities?**
Yes, sites maintain a safety buffer of **14 km to 23 km** from major residential centers like Titusville and Lompoc. This protects the public from potential launch failures and mitigates the impact of extreme noise and sonic booms.

### Data Analysis Key Findings

*   **Coastline Proximity:** Every launch site is positioned within **0.8 km to 7.6 km** of the coast to prioritize safety and stage disposal over open water.
*   **Infrastructure Synergy:** There is a high correlation between launch site location and heavy transport infrastructure, specifically railways (**0.7 km - 1.3 km**) and highways (**0.04 km - 5.8 km**).
*   **Safety Buffers:** A consistent exclusion zone of **14 km to 23 km** is maintained from urban populations to manage acoustic impact and debris risk.
*   **Site Specifics:**
    *   **KSC LC-39A** is the most integrated with road infrastructure, being only **0.04 km** from Kennedy Parkway North.
    *   **VAFB SLC-4E** is the most isolated from highways (**5.8 km**) but remains close to essential rail lines.

### Insights or Next Steps

*   **Next Step:** Perform a temporal analysis to see if landing zone locations (for reusable boosters) follow similar geographical constraints regarding proximity to infrastructure and distance from cities.
*   **Insight:** The geographical footprint of a SpaceX launch site is a calculated balance between "Logistical Access" (very close to rail/road) and "Public Risk Mitigation" (far from cities, adjacent to water).


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
